# O(3), Spherical Harmonics, and Cartesian Tensors

**6.7970/8.750 Symmetry and its Application to Machine Learning**

Companion notebook for the [O(3) and Spherical Harmonics notes](https://symm4ml.mit.edu/symm4ml_s26/notes/o3-and-spherical-harmonics). We will:

1. Review SO(3) irreps and CG coefficients
2. Extend to O(3) by adding parity
3. Decompose Cartesian tensors using index symmetries
4. Branch O(3) irreps under point group subgroups
5. Construct spherical harmonics from CG coefficients

In [ ]:
%%capture
!pip install torch pymatgen
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from symm4ml import groups, linalg, rep, vis, vib_modes, lie, so3, grids

---
## 1. SO(3) Irreps and CG Coefficients

### Build irreps via tensor products

In [ ]:
so3_irreps = lie.infer_irreps_from_tensor_products(so3.so3_gens, 5)
print(f'Found {len(so3_irreps)} SO(3) irreps:')
for l, ir in enumerate(so3_irreps):
    print(f'  l={l}: dim {ir.shape[1]}, L\u00b2 = {(ir @ ir).sum(0)[0,0]:.1f}')

### CG coefficients and selection rule

CG coefficients are non-zero only when $|l_1 - l_2| \leq l_3 \leq l_1 + l_2$ (triangle inequality):

In [ ]:
L0, L1, L2, L3, L4 = [so3_irreps[i] for i in range(5)]

# Non-zero: |1-2| = 1 <= 3 <= 1+2 = 3 ✓
cg_L1_L2_L3 = so3.clebsch_gordan(L1, L2, L3)
print(f'CG(l=1, l=2 -> l=3): shape {cg_L1_L2_L3.shape}, zero? {np.allclose(cg_L1_L2_L3, 0)}')

# Zero: |2-1| = 1 <= 4? No, 4 > 2+1 = 3
cg_L2_L1_L4 = so3.clebsch_gordan(L2, L1, L4)
print(f'CG(l=2, l=1 -> l=4): shape {cg_L2_L1_L4.shape}, zero? {np.allclose(cg_L2_L1_L4, 0)}')

### CG normalization

CG coefficients are normalized so that all output paths from two unit-norm inputs produce unit norm:

In [ ]:
# l=1 \otimes l=1 = l=0 + l=1 + l=2
cg_L1_L1_L0 = so3.clebsch_gordan(L1, L1, L0)
cg_L1_L1_L1 = so3.clebsch_gordan(L1, L1, L1)
cg_L1_L1_L2 = so3.clebsch_gordan(L1, L1, L2)

# Stack all CG paths and contract with unit-norm inputs
x = np.array([1, 0, 0], dtype=float)  # unit vector
y = np.array([0, 1, 0], dtype=float)

out_L0 = np.einsum('ijk,i,j->k', cg_L1_L1_L0, x, y)
out_L1 = np.einsum('ijk,i,j->k', cg_L1_L1_L1, x, y)
out_L2 = np.einsum('ijk,i,j->k', cg_L1_L1_L2, x, y)

total_norm_sq = np.linalg.norm(out_L0)**2 + np.linalg.norm(out_L1)**2 + np.linalg.norm(out_L2)**2
print(f'||out_L0||\u00b2 + ||out_L1||\u00b2 + ||out_L2||\u00b2 = {total_norm_sq:.4f} (should be 1.0)')

---
## 2. From SO(3) to O(3): Adding Parity

$O(3) = SO(3) \times \{I, -I\}$. Each SO(3) irrep $l$ gives two O(3) irreps: even parity ($le$) and odd parity ($lo$).

In [ ]:
# Sample random O(3) elements (rotations with random signs)
np.random.seed(42)
N = 10
rand_rots = np.random.rand(N, 3) * 2 * np.pi
rand_signs = np.array([1] * (N//2) + [-1] * (N//2))

# Build O(3) irreps: for each l, create even and odd parity versions
o3_irreps = []
o3_irrep_labels = []
for l, so3_ir in enumerate(so3_irreps):
    for p_label, p in [('e', 1), ('o', -1)]:
        rot_params, sign_params = so3.vec_3d_rep_to_rot_and_sign_params(
            np.array([s * so3.axis_angle_to_matrix(r) for s, r in zip(rand_signs, rand_rots)])
        )
        o3_ir = so3.o3_irrep(so3_ir, rot_params, sign_params, parity=p)
        o3_irreps.append(o3_ir)
        o3_irrep_labels.append(f'{l}{p_label}')

print('O(3) irreps:', o3_irrep_labels)
print('Dimensions:', [ir.shape[1] for ir in o3_irreps])

### Tensor products in O(3)

Parity multiplication: even $\times$ even = even, even $\times$ odd = odd, odd $\times$ odd = even.

In [ ]:
# 1o \otimes 1o
o3_1o = o3_irreps[o3_irrep_labels.index('1o')]
o3_1o_1o = rep.tensor_product(o3_1o, o3_1o)

cob = [linalg.infer_change_of_basis(ir, o3_1o_1o) for ir in o3_irreps]
print('1o \u2297 1o decomposes as:')
for label, c in zip(o3_irrep_labels, cob):
    if c.shape[0] > 0:
        print(f'  {label}: {c.shape[0]} copies')
print('\n= 0e \u2295 1e \u2295 2e  (all even parity, as expected from odd \u00d7 odd = even)')

---
## 3. Cartesian Tensors and Index Symmetries

### Symmetric rank-2 tensor: $ij = ji$

In [ ]:
# Generate the permutation and sign group for ij=ji
perm_rep, sign_rep = grids.formula_to_perm_and_sign_group('ij=ji')
print('Permutation group:')
print(perm_rep)

# Compute the invariant subspace basis
new_basis, new_proj = grids.perm_and_sign_to_tensor_basis_and_proj(perm_rep, sign_rep, [3, 3])
print(f'\nSymmetric subspace dimension: {new_basis.shape[0]} (from 9 -> 6)')

# Project O(3) tensor rep onto symmetric subspace and decompose
sub = np.einsum('ij,njk,kl->nil', new_basis, o3_1o_1o, new_basis.T)
cob = [linalg.infer_change_of_basis(ir, sub) for ir in o3_irreps]
print('\nSymmetric tensor decomposes as:')
for label, c in zip(o3_irrep_labels, cob):
    if c.shape[0] > 0:
        print(f'  {label}: {c.shape[0]} copies')
print('\n= 0e \u2295 2e  (the antisymmetric l=1 part vanished!)')

### Fully symmetric rank-3 tensor: $ijk = jik = ikj$

In [ ]:
o3_1o_1o_1o = rep.tensor_product(o3_1o_1o, o3_1o)

perm_rep3, sign_rep3 = grids.formula_to_perm_and_sign_group('ijk=jik=ikj')
new_basis3, new_proj3 = grids.perm_and_sign_to_tensor_basis_and_proj(perm_rep3, sign_rep3, [3, 3, 3])
print(f'Fully symmetric rank-3 subspace dimension: {new_basis3.shape[0]} (from 27 -> 10)')

sub3 = np.einsum('ij,njk,kl->nil', new_basis3, o3_1o_1o_1o, new_basis3.T)
cob3 = [linalg.infer_change_of_basis(ir, sub3) for ir in o3_irreps]
print('\nFully symmetric rank-3 tensor decomposes as:')
for label, c in zip(o3_irrep_labels, cob3):
    if c.shape[0] > 0:
        print(f'  {label}: {c.shape[0]} copies')
print('\n= 1o \u2295 3o')

---
## 4. Branching Under Point Groups

Every point group is a subgroup of O(3). O(3) irreps branch (decompose) into irreps of the subgroup.

In [ ]:
from pymatgen.symmetry.groups import PointGroup

def o3_branch(so3_irreps, sub, sub_irreps, sub_irrep_labels, max_l=4):
    """Branch O(3) irreps under a point group subgroup."""
    rot_params, sign_params = so3.vec_3d_rep_to_rot_and_sign_params(sub)
    o3_irs = [
        so3.o3_irrep(so3_irreps[l], rot_params, sign_params, parity=p)
        for l in range(min(max_l, len(so3_irreps))) for p in [1, -1]
    ]
    o3_labels = [f'{l}{p}' for l in range(max_l) for p in ['e', 'o']]
    branch = vib_modes.branching_change_of_basis(o3_irs, sub_irreps, np.arange(len(sub)))
    
    # Print table
    header = '\t'.join(sub_irrep_labels)
    print(f'O(3)\t{header}')
    print('-' * (8 + 8 * len(sub_irrep_labels)))
    for i, label in enumerate(o3_labels):
        counts = [branch[i][j].shape[0] for j in range(len(sub_irrep_labels))]
        row = '\t'.join(str(c) if c > 0 else '.' for c in counts)
        print(f'{label}\t{row}')
    return branch

### Branching under $T_d$ (tetrahedral)

In [ ]:
# Build Td group
Td_group = PointGroup(vib_modes.point_group_dict['Td'])
Td = np.stack([op.rotation_matrix for op in Td_group.symmetry_ops], axis=0)
Td_table = groups.make_multiplication_table(Td)
Td_irreps = rep.infer_irreps(Td_table)

# Reorder to match standard character table: A1, A2, E, T1, T2
Td_irrep_labels = ['A1', 'A2', 'E', 'T1', 'T2']
# (Reordering may be needed depending on infer_irreps output)
Td_dims = [ir.shape[1] for ir in Td_irreps]
print(f'Td irrep dimensions: {Td_dims}')
print(f'Labels: {Td_irrep_labels}')
print()

b_Td = o3_branch(so3_irreps, Td, Td_irreps, Td_irrep_labels)

### Point group constraints on tensor components

Only components that branch to $A_1$ (trivial irrep) can be non-zero for a system with that symmetry.

In [ ]:
# For a symmetric rank-2 tensor (0e + 2e) under Td:
# 0e -> A1 (1 component survives)
# 2e -> E + T2 (5 components, none are A1 -> all vanish!)
print('Symmetric rank-2 tensor under Td:')
print('  0e -> A1: 1 independent component (isotropic part)')
print('  2e -> E + T2: 0 independent components (forced to zero by symmetry)')
print('  Total: 1 of 6 components survive')

---
## 5. Spherical Harmonics from CG Coefficients

The CG change-of-basis from $1o \otimes 1o \to 2e$ gives the degree-2 spherical harmonics as polynomials in $(x, y, z)$:

In [ ]:
# Change of basis: 1o x 1o -> 2e
o3_2e = o3_irreps[o3_irrep_labels.index('2e')]
cob = linalg.infer_change_of_basis(o3_1o_1o, o3_2e)
print(f'Change of basis shape: {cob.shape}')

# Reshape to (3, 3, 5): (l1 components) x (l2 components) x (l=2 components)
cg_mat = cob[0].reshape(3, 3, 5)

# Visualize: 5 matrices, each showing how two vectors couple to one l=2 component
fig = vis.plot_matrices(np.einsum('ijk->kij', cg_mat.real), cmap='RdBu')
plt.suptitle('CG coefficients for 1o \u2297 1o \u2192 2e\n'
             '(5 matrices = 5 spherical harmonic components Y\u2082\u1d50)', y=1.05)
plt.show()

In [ ]:
# Read off the polynomials: contract CG with symbolic (x,y,z)
import sympy as sp
x, y, z = sp.symbols('x y z')
vec = sp.Matrix([x, y, z])

tensor = np.einsum('ijk->kij', cg_mat.real).round(4)
print('Degree-2 spherical harmonics as polynomials:\n')
for m in range(5):
    T = sp.Matrix(tensor[m])
    poly = (vec.T @ T @ vec)[0, 0]
    print(f'  Y_2^{m-2}: {sp.simplify(poly)}')

print('\nThese are the 5 independent components of a symmetric traceless rank-2 tensor.')
print('On the unit sphere (r=1), these are the standard real spherical harmonics Y_2^m.')

### Degree-3 spherical harmonics from $1o \otimes 1o \otimes 1o \to 3o$

In [ ]:
o3_3o = o3_irreps[o3_irrep_labels.index('3o')]
cob3 = linalg.infer_change_of_basis(o3_1o_1o_1o, o3_3o)
print(f'Change of basis shape: {cob3.shape}')

cg_mat3 = cob3[0].reshape(3, 3, 3, 7)
tensor3 = np.einsum('ijkl->lijk', cg_mat3.real).round(4)

print('\nDegree-3 spherical harmonics as polynomials:\n')
for m in range(7):
    T = tensor3[m]
    poly = sum(T[i,j,k] * [x,y,z][i] * [x,y,z][j] * [x,y,z][k]
              for i in range(3) for j in range(3) for k in range(3))
    print(f'  Y_3^{m-3}: {sp.simplify(poly)}')